In [1]:
import pandas as pd

In [99]:
input_file = "data/sosedushka.xlsx"

sheets = pd.read_excel(input_file, sheet_name=None)

#### Collect Russian column names and values

In [37]:
column_names = []

for sheet, data in sheets.items():
    print(sheet)
    column_names += data.columns.tolist()

README
fact_sales
fact_sales_dirty
dim_products
dim_stores
dim_customers
dim_suppliers


In [56]:
values = []

for sheet, data in sheets.items():
    if sheet != 'README' and sheet != 'fact_sales_dirty':
        for col_name in data.select_dtypes(include=['object', 'string']).columns:
            values += sheets[sheet][col_name].unique().tolist()

values = list(set(values))

In [52]:
print(len(values))

1075


In [60]:
with open('python/russian_values.txt', 'w') as f:
    for word in values:
        f.write(word.lower() + '\n')

In [53]:
with open('python/russian_columns.txt', 'w') as f:
    for word in column_names:
        f.write(word.lower().replace(' ', '_') + '\n')

#### Russian -> English dictionaries


In [111]:
value_translation = {}

with open('python/english_values.txt', 'r') as f:
    for val, translation in zip(values, f.read().splitlines()):
        value_translation[val.lower()] = translation

len(value_translation)

1075

In [101]:
value_translation['лук репчатый 1кг']

'onions 1kg'

In [108]:
column_translation = {}

with open('python/english_columns.txt', 'r') as f:
    for val, translation in zip(column_names, f.read().splitlines()):
        column_translation[val.lower()] = translation

In [103]:
column_translation['фио']

'full_name'

#### Rebuilding the file with the columns and text values translated into English

In [112]:
output_file = "data/dear_neighbour.xlsx"

selected_sheets = [
    "fact_sales",
    "dim_products",
    "dim_stores",
    "dim_customers",
    "dim_suppliers"
]

translated_sheets = {}

for sheet_name in selected_sheets:
    df = sheets[sheet_name].copy()

    df.rename(
       columns=lambda col: column_translation.get(col.strip().replace(' ', '_').lower(), col),
        inplace=True
    )

    for column in df.columns:
        if df[column].dtype == "string":
            df[column] = df[column].map(
                lambda x: value_translation.get(x.lower(), x)
            )

    translated_sheets[sheet_name] = df

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in translated_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)